In [ ]:
import os
import urllib.request
import zipfile
import shutil
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

%cd /content
!rm -rf MUSAE
!git clone https://github.com/benedekrozemberczki/MUSAE.git
%cd /content/MUSAE

!pip install -q networkx texttable scipy numpy scikit-learn gensim

!rm -rf input output
os.makedirs("input/edges", exist_ok=True)
os.makedirs("input/features", exist_ok=True)
os.makedirs("input/target", exist_ok=True)
os.makedirs("output", exist_ok=True)

print("1. Đang tải dữ liệu gốc từ Stanford (khoảng 20MB)...")
zip_path = "/content/twitch.zip"
if not os.path.exists(zip_path):
    urllib.request.urlretrieve("https://snap.stanford.edu/data/twitch.zip", zip_path)

print("2. Đang định vị và trích xuất dữ liệu...")
with zipfile.ZipFile(zip_path, "r") as z:
    for name in z.namelist():
        if "PT" in name.upper() and "edges.csv" in name:
            with z.open(name) as src, open("input/edges/twitch_pt.csv", "wb") as dst:
                shutil.copyfileobj(src, dst)
        elif "PT" in name.upper() and "features.json" in name:
            with z.open(name) as src, open("input/features/twitch_pt.json", "wb") as dst:
                shutil.copyfileobj(src, dst)
        elif "PT" in name.upper() and "target.csv" in name:
            with z.open(name) as src, open("input/target/twitch_pt.csv", "wb") as dst:
                shutil.copyfileobj(src, dst)

print("   -> Đã trích xuất xong! Bắt đầu huấn luyện...")

print("3. Đang chạy thuật toán MUSAE (sẽ mất khoảng 1-2 phút)...")
!python src/main.py --graph-input input/edges/twitch_pt.csv --features-input input/features/twitch_pt.json --output output/twitch_pt_embeddings.csv

print("4. Bắt đầu chấm điểm mô hình phân loại...")
try:
    embeddings_df = pd.read_csv("output/twitch_pt_embeddings.csv")
    X = embeddings_df.drop('id', axis=1).values

    target_df = pd.read_csv("input/target/twitch_pt.csv")
    y = target_df['target'].values

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    clf = LogisticRegression(max_iter=1000)
    clf.fit(X_train, y_train)

    print("\n" + "="*55)
    print("  BẢNG ĐIỂM DỰ ĐOÁN NGÔN NGỮ NHẠY CẢM (TWITCH PT)")
    print("="*55)
    print(classification_report(y_test, clf.predict(X_test)))
except Exception as e:
    print(f"\n[LỖI] Đã xảy ra vấn đề: {e}")

/content
Cloning into 'MUSAE'...
remote: Enumerating objects: 672, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 672 (delta 2), reused 0 (delta 0), pack-reused 666 (from 1)
Receiving objects: 100% (672/672), 19.88 MiB | 8.59 MiB/s, done.
Resolving deltas: 100% (358/358), done.
/content/MUSAE
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 14.4 MB/s eta 0:00:00
1. Đang tải dữ liệu gốc từ Stanford (khoảng 20MB)...
2. Đang định vị và trích xuất dữ liệu...
   -> Đã trích xuất xong! Bắt đầu huấn luyện...
3. Đang chạy thuật toán MUSAE (sẽ mất khoảng 1-2 phút)...
+---------------------+---------------------------------+
|          P          |               1.0               |
+=====================+=================================+
| Q                   | 1                               |
+---------------------+---------------------------------+
| Alpha               | 0.050                           |
+--------------